```
# =============================================================================
# Agentic RAG — Query Splitting + Query Rewriting
# Deployment-level implementation
# =============================================================================
#
# Flow:
#   START → decompose → agent → retriever → grade_documents
#                                               ├─→ generate → END
#                                               ├─→ rewrite  → agent (loop)
#                                               └─→ END (all failed + ceiling hit)
#
# Key behaviours:
#   - Complex questions are split into sub-questions before retrieval
#   - Each sub-question is retrieved and graded independently
#   - Only failed sub-questions are rewritten — successful ones are preserved
#   - A rewrite ceiling (MAX_REWRITES) prevents infinite loops
#   - Partial results are used for generation when some sub-questions keep failing
#   - Simple questions (no real sub-questions) flow through unchanged
# =============================================================================


In [13]:
# ── Imports ───────────────────────────────────────────────────────────────────
import logging
import os
from typing import Annotated, Literal, Sequence

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.vectorstores import FAISS
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

In [14]:
# ── Logging ───────────────────────────────────────────────────────────────────
import logging
from logging.handlers import RotatingFileHandler

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(name)s — %(message)s")

# Console handler — keeps logs visible in terminal
console_handler = logging.StreamHandler()
console_handler.setFormatter(formatter)

# File handler — persists logs to disk, rotates at 5MB, keeps last 3 files
file_handler = RotatingFileHandler(
    filename="agent.log",
    maxBytes=5 * 1024 * 1024,  # 5 MB per file
    backupCount=3,              # keep agent.log, agent.log.1, agent.log.2
    encoding="utf-8",
)
file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

# ── Configuration ─────────────────────────────────────────────────────────────
load_dotenv()
# load_dotenv() already populates os.environ from .env.
# No need to re-assign os.environ["OPENAI_API_KEY"].

MAX_REWRITES = 3        # maximum rewrite attempts per sub-question before giving up
MAX_SUB_QUESTIONS = 5   # cap on decomposition to prevent overly broad splits

try:
    llm = init_chat_model("openai:gpt-4o")
    logger.info("LLM initialised: gpt-4o")
except Exception as e:
    logger.exception("Failed to initialise LLM.")
    raise

2026-02-19 21:26:50,486 [INFO] __main__ — LLM initialised: gpt-4o
2026-02-19 21:26:50,486 [INFO] LLM initialised: gpt-4o


```

So the flow is now:

Agent decides to use tool
        ↓
retrieve_from_vectorstore(query) is called by the agent
        ↓
retriever.invoke(query)  ← MMR handles diversity
        ↓
similarity_search_with_relevance_scores(query) ← gets scores for quality filtering
        ↓
score_lookup filters out low-quality MMR chunks
        ↓
returns formatted string back to agent
```

In [15]:
# =============================================================================
# 1. LangGraph State
# =============================================================================

class AgentState(TypedDict):
    """
    State passed between every node in the LangGraph workflow.

    messages:        Full conversation history, merged by add_messages reducer.
    sub_questions:   List of sub-questions produced by the decompose node.
                     On rewrite passes, contains only the failed sub-questions.
    retrieved_docs:  Maps each sub-question → its best retrieved content string.
                     Accumulates across retrieval passes so successful results
                     are never overwritten.
    failed_questions: Sub-questions whose retrieved docs were graded as not
                     relevant.  Populated by grade_documents, consumed by rewrite.
    rewrite_count:   Number of rewrite cycles completed.  Compared against
                     MAX_REWRITES to break out of loops.
    """
    messages:          Annotated[Sequence[BaseMessage], add_messages]
    sub_questions:     list[str]
    retrieved_docs:    dict[str, str]   # sub-question → retrieved content
    failed_questions:  list[str]        # sub-questions that failed grading
    rewrite_count:     int


In [16]:

# =============================================================================
# 2. Vectorstore builder
# =============================================================================

def build_vectorstore_from_urls(
    urls: list[str],
    faiss_index_path: str,
    chunk_size: int = 1000,
    chunk_overlap: int = 100,
    embedding_model: str = "text-embedding-3-small",
    embedding_chunk_size: int = 200,
) -> FAISS:
    """
    Load a FAISS index from disk if it exists, otherwise scrape the given URLs,
    embed the chunks, and persist the index for future runs.

    Args:
        urls:                 Web pages to load when building from scratch.
        faiss_index_path:     Directory path to save/load the FAISS index.
        chunk_size:           Character size of each document chunk.
        chunk_overlap:        Overlap between consecutive chunks.
        embedding_model:      OpenAI embedding model name.
        embedding_chunk_size: Batch size for the embedding API calls.

    Returns:
        A FAISS vectorstore instance ready for similarity search.

    Raises:
        ValueError: If urls is empty when building from scratch.
        RuntimeError: If the index cannot be loaded or built.
    """
    if not faiss_index_path:
        raise ValueError("faiss_index_path must not be empty.")

    embedding = OpenAIEmbeddings(
        model=embedding_model,
        chunk_size=embedding_chunk_size,
    )

    # ── Load from disk (cache hit) ────────────────────────────────────────────
    if os.path.exists(faiss_index_path):
        try:
            vs = FAISS.load_local(
                faiss_index_path,
                embeddings=embedding,
                allow_dangerous_deserialization=True,
            )
            logger.info("VectorStore loaded from disk: %s", faiss_index_path)
            return vs
        except Exception as e:
            logger.warning(
                "Failed to load FAISS index from %s (%s). Rebuilding...",
                faiss_index_path, e
            )

    # ── Build from URLs (cache miss) ──────────────────────────────────────────
    if not urls:
        raise ValueError(
            f"No FAISS index found at '{faiss_index_path}' and no URLs provided to build one."
        )

    logger.info("Building FAISS index from %d URL(s)...", len(urls))

    try:
        docs = [item for url in urls for item in WebBaseLoader(url).load()]
    except Exception as e:
        raise RuntimeError(f"Failed to load documents from URLs: {e}") from e

    if not docs:
        raise RuntimeError("No documents were loaded from the provided URLs.")

    doc_splits = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        add_start_index=True,   # stores char offset in metadata for traceability
    ).split_documents(docs)

    if not doc_splits:
        raise RuntimeError("Text splitter produced zero chunks from the loaded documents.")

    try:
        vectorstore = FAISS.from_documents(documents=doc_splits, embedding=embedding)
        vectorstore.save_local(faiss_index_path)
    except Exception as e:
        raise RuntimeError(f"Failed to build or save FAISS index: {e}") from e

    logger.info(
        "VectorStore built and saved: %s (%d chunks)", faiss_index_path, len(doc_splits)
    )
    return vectorstore



In [17]:

# =============================================================================
# 3. Custom retriever tool factory
# =============================================================================

def create_custom_retriever_tool(
    vectorstore: FAISS,
    tool_name: str,
    tool_description: str,
    threshold: float = 0.30,
    k: int = 5,
    fetch_k: int = 20,
    lambda_mult: float = 0.6,
):
    """
    Returns a @tool that performs two-stage retrieval:
      Stage 1 — MMR retrieval for diversity-aware candidate selection.
      Stage 2 — Relevance score filtering to drop low-quality chunks.

    The retriever and vectorstore are captured in a closure so each tool
    instance operates independently with its own configuration.

    Args:
        vectorstore:      The FAISS index to search.
        tool_name:        Name the LLM uses to identify this tool.
        tool_description: Description shown to the LLM for tool selection.
        threshold:        Minimum relevance score (0–1) to keep a chunk.
        k:                Number of final chunks to return.
        fetch_k:          Candidate pool size for MMR selection.
        lambda_mult:      MMR diversity/relevance trade-off (0=diversity, 1=relevance).

    Returns:
        A LangChain @tool callable.
    """
    if not tool_name:
        raise ValueError("tool_name must not be empty.")
    if not (0.0 <= threshold <= 1.0):
        raise ValueError(f"threshold must be between 0 and 1, got {threshold}.")
    if k < 1 or fetch_k < k:
        raise ValueError("k must be >= 1 and fetch_k must be >= k.")

    # MMR retriever — constructed once, reused on every tool call via closure
    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": k, "fetch_k": fetch_k, "lambda_mult": lambda_mult},
    )

    @tool(tool_name)
    def retrieve_from_vectorstore(query: str) -> str:
        """Placeholder — overridden by tool_description below."""

        if not query or not query.strip():
            return "Empty query — nothing to retrieve."

        try:
            # Stage 1: MMR gives a diverse, relevant candidate set
            mmr_docs = retriever.invoke(query)
        except Exception as e:
            logger.error("MMR retrieval failed for query '%s': %s", query, e)
            return "Retrieval failed due to an internal error."

        if not mmr_docs:
            return "No documents found in the local vector store."

        try:
            # Stage 2: Fetch scores — MMR does not return scores itself
            docs_with_scores = vectorstore.similarity_search_with_relevance_scores(
                query, k=k
            )
        except Exception as e:
            logger.error("Relevance scoring failed for query '%s': %s", query, e)
            # Fall back to returning MMR docs without score filtering
            results = [
                f"[Chunk {i}] Source: {doc.metadata.get('source', 'Unknown')}\n{doc.page_content}"
                for i, doc in enumerate(mmr_docs, 1)
            ]
            return "\n\n---\n\n".join(results)

        # Build lookup: page_content → relevance score
        score_lookup: dict[str, float] = {
            doc.page_content: float(score)
            for doc, score in docs_with_scores
        }

        # Keep only MMR docs whose similarity score clears the threshold
        filtered = [
            (doc, score_lookup.get(doc.page_content, 0.0))
            for doc in mmr_docs
            if score_lookup.get(doc.page_content, 0.0) >= threshold
        ]

        if not filtered:
            return "No relevant documents found in the local vector store."

        results = [
            f"[Chunk {i}] Score: {score:.2f} | Source: {doc.metadata.get('source', 'Unknown')}\n{doc.page_content}"
            for i, (doc, score) in enumerate(filtered, 1)
        ]
        return "\n\n---\n\n".join(results)

    retrieve_from_vectorstore.description = tool_description
    return retrieve_from_vectorstore


In [18]:
# =============================================================================
# 4. Build vectorstores + tools
# =============================================================================

langgraph_vs = build_vectorstore_from_urls(
    urls=[
        "https://langchain-ai.github.io/langgraph/tutorials/introduction/#building-with-langgraph",
        "https://langchain-ai.github.io/langgraph/tutorials/workflows/",
        "https://langchain-ai.github.io/langgraph/how-tos/map-reduce/",
    ],
    faiss_index_path="langgraph_blog_faiss_index",
)

langchain_vs = build_vectorstore_from_urls(
    urls=[
        "https://python.langchain.com/docs/tutorials/",
        "https://python.langchain.com/docs/tutorials/chatbot/",
        "https://python.langchain.com/docs/tutorials/qa_chat_history/",
    ],
    faiss_index_path="langchain_blog_faiss_index",
)

retriever_tool = create_custom_retriever_tool(
    langgraph_vs,
    tool_name="retriever_vector_db_blog",
    tool_description="Search and return information about LangGraph",
    threshold=0.30,
)

retriever_tool_langchain = create_custom_retriever_tool(
    langchain_vs,
    tool_name="retriever_vector_langchain_blog",
    tool_description="Search and return information about LangChain",
    threshold=0.35,
    lambda_mult=0.7,
)

# Wikipedia as a fallback when local docs are insufficient
wikipedia_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=3,
        doc_content_chars_max=2000,
    )
)

tools = [retriever_tool, retriever_tool_langchain, wikipedia_tool]


2026-02-19 21:26:50,906 [INFO] __main__ — VectorStore loaded from disk: langgraph_blog_faiss_index
2026-02-19 21:26:50,906 [INFO] VectorStore loaded from disk: langgraph_blog_faiss_index
2026-02-19 21:26:51,264 [INFO] __main__ — VectorStore loaded from disk: langchain_blog_faiss_index
2026-02-19 21:26:51,264 [INFO] VectorStore loaded from disk: langchain_blog_faiss_index


In [19]:
# =============================================================================
# 5. Shared chains (built once at module level — not rebuilt per node call)
# =============================================================================

# ── Decompose chain ───────────────────────────────────────────────────────────
_decompose_prompt = ChatPromptTemplate.from_messages([
    (
        "human",
        "You are a question decomposition assistant.\n\n"
        "Break the following question into at most {max_sub} smaller, self-contained "
        "sub-questions that together cover the full intent of the original question.\n\n"
        "Rules:\n"
        "- If the question is already simple and specific, return it as a single item.\n"
        "- Each sub-question must be independently answerable.\n"
        "- Do not add sub-questions that are not implied by the original.\n"
        "- Return ONLY a numbered list, one sub-question per line, no extra text.\n\n"
        "Question: {question}",
    )
])
_decompose_chain = _decompose_prompt | llm | StrOutputParser()


# ── Relevance grading ─────────────────────────────────────────────────────────
class RelevanceGrade(BaseModel):
    """Binary relevance score for a retrieved document."""
    binary_score: str = Field(description="Relevance score: 'yes' or 'no'")

_grade_prompt = PromptTemplate(
    template=(
        "You are a grader assessing whether a retrieved document is relevant "
        "to a specific question.\n\n"
        "Retrieved document:\n{context}\n\n"
        "Question: {question}\n\n"
        "Grade 'yes' if the document contains keywords or semantic meaning related "
        "to the question. Grade 'no' otherwise.\n"
        "Return only 'yes' or 'no'."
    ),
    input_variables=["context", "question"],
)
_grade_chain = _grade_prompt | llm.with_structured_output(RelevanceGrade)


# ── Rewrite chain ─────────────────────────────────────────────────────────────
_rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "human",
        "You are a query rewriting assistant.\n\n"
        "The following question failed to retrieve relevant documents.\n"
        "Rewrite it to be clearer, more specific, and more likely to match "
        "technical documentation.\n\n"
        "Original question: {question}\n\n"
        "Return only the rewritten question, no explanation.",
    )
])
_rewrite_chain = _rewrite_prompt | llm | StrOutputParser()


# ── RAG generation chain ──────────────────────────────────────────────────────
_generate_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an assistant for question-answering tasks.\n"
        "Use the retrieved context below to answer the question.\n"
        "The context is organised by sub-question — use all of it.\n"
        "If some sub-questions have no context, acknowledge what is unknown.\n"
        "Keep the answer concise (five sentences maximum).\n\n"
        "Context:\n{context}",
    ),
    ("human", "{question}"),
])
_rag_chain = _generate_prompt | llm | StrOutputParser()


# ── Agent model (tools bound once) ────────────────────────────────────────────
_agent_model = llm.bind_tools(tools)



In [20]:

# =============================================================================
# 6. Graph nodes
# =============================================================================

# ── Node 1: decompose ─────────────────────────────────────────────────────────
def decompose(state: AgentState) -> dict:
    """
    Split the original user question into smaller sub-questions.

    Behaviour:
    - Calls the decompose LLM chain with a cap of MAX_SUB_QUESTIONS.
    - Parses the numbered list output into a clean list of strings.
    - If parsing fails or returns nothing, falls back to the original question
      as a single-item list so the rest of the graph is unaffected.
    - Simple questions that don't need splitting are returned as-is by the LLM.

    State changes:
        sub_questions     ← populated with decomposed queries
        retrieved_docs    ← initialised as empty dict
        failed_questions  ← initialised as empty list
        rewrite_count     ← initialised to 0
    """
    logger.info("---DECOMPOSE---")

    question = state["messages"][0].content

    try:
        raw = _decompose_chain.invoke({
            "question": question,
            "max_sub": MAX_SUB_QUESTIONS,
        })
    except Exception as e:
        logger.error("Decompose chain failed: %s. Falling back to original question.", e)
        raw = f"1. {question}"

    # Parse "1. question\n2. question\n..." into a clean list
    sub_questions: list[str] = []
    for line in raw.strip().splitlines(): 
        line = line.strip()
        if not line:
            continue
        # Strip leading "1." / "1)" / "-" markers
        for sep in [". ", ") ", "- "]:
            if len(line) > 2 and line[1:3] == sep[:2]:
                line = line[3:].strip()
                break
        if line:
            sub_questions.append(line)

    # Fallback: if parsing produced nothing, use the original question
    if not sub_questions:
        logger.warning("Decompose produced no sub-questions. Using original question.")
        sub_questions = [question]

    logger.info("Decomposed into %d sub-question(s): %s", len(sub_questions), sub_questions)

    return {
        "sub_questions":    sub_questions,
        "retrieved_docs":   {},     # initialise accumulator
        "failed_questions": [],     # initialise failure tracker
        "rewrite_count":    0,
    }


# ── Node 2: agent ─────────────────────────────────────────────────────────────
def agent(state: AgentState) -> dict:
    """
    Invoke the LLM for each pending sub-question to decide which tool to call.

    On the first pass, all sub_questions are pending.
    On rewrite passes, only failed_questions are pending (already-successful
    sub-questions keep their results in retrieved_docs untouched).

    The agent is given one sub-question at a time so tool selection stays
    focused. Responses are appended to messages for the ToolNode to process.

    State changes:
        messages ← one AIMessage per sub-question (may include tool_calls)
    """
    logger.info("---CALL AGENT---")

    # On first pass use sub_questions; on rewrite passes use failed_questions
    failed = state.get("failed_questions", [])
    pending = failed if failed else state.get("sub_questions", [])

    if not pending:
        logger.warning("Agent called with no pending sub-questions.")
        return {"messages": []}

    responses = []
    for sub_q in pending:
        try:
            response = _agent_model.invoke([HumanMessage(content=sub_q)])
            responses.append(response)
            logger.info("Agent processed sub-question: '%s'", sub_q)
        except Exception as e:
            logger.error("Agent failed for sub-question '%s': %s", sub_q, e)
            # Append a fallback message so the graph doesn't stall
            responses.append(HumanMessage(content=f"[ERROR] Could not process: {sub_q}"))

    return {"messages": responses}


# ── Node 3: retriever (ToolNode) ───────────────────────────────────────────────
# ToolNode is a prebuilt node — it reads tool_calls from the last AIMessage,
# executes the corresponding tool, and appends a ToolMessage with the result.
_tool_node = ToolNode(tools)

# ── Node 4: grade  (FIX 1 — was a misused edge function in v2) ───────────────
def grade(state: AgentState) -> dict:
    """
    Grade each sub-question's retrieved content independently and write
    results back to state.

    THIS IS A NODE, not an edge function. It returns a dict to update state.
    The actual routing decision is made by route_after_grade (edge function)
    which reads the updated state after this node completes.

    Why the split?  In LangGraph, edge functions are pure routing functions —
    they return a string, not a dict. Any attempt to mutate state inside an
    edge function is silently discarded. This was the root cause of the
    failed_questions always being empty in v2.

    Steps:
      1. Extract the latest ToolMessages and map them to pending sub-questions.
      2. Update retrieved_docs (only overwrite empty/missing entries).
      3. Grade each sub-question against its retrieved content.
      4. Write failed_questions and updated retrieved_docs back to state.

    Returns:
        failed_questions: sub-questions that failed or had no content
        retrieved_docs:   updated accumulator
    """
    logger.info("---GRADE---")

    sub_questions:  list[str]  = state.get("sub_questions", [])
    retrieved_docs: dict       = dict(state.get("retrieved_docs", {}))  # copy to avoid mutation
    failed_in_prev: list[str]  = state.get("failed_questions", [])

    # Pending = failed sub-questions on rewrite passes, all on first pass
    pending = failed_in_prev if failed_in_prev else sub_questions

    # ── Step 1: map latest ToolMessages → pending sub-questions ───────────────
    messages    = list(state["messages"])
    tool_msgs   = [m for m in messages if getattr(m, "type", None) == "tool"]

    # ToolNode appends one ToolMessage per tool_call in order.
    # We match by position against the pending list for this pass.
    no_result_sentinel = "No relevant documents found in the local vector store."
    for i, sub_q in enumerate(pending):
        content = tool_msgs[i].content.strip() if i < len(tool_msgs) else ""
        # Only update if we received actual content (preserve prior successes)
        if content and content != no_result_sentinel:
            retrieved_docs[sub_q] = content
        elif sub_q not in retrieved_docs:
            retrieved_docs[sub_q] = ""  # mark as attempted-but-empty

    # ── Step 2: grade every sub-question ──────────────────────────────────────
    failed: list[str] = []
    passed: list[str] = []

    for sub_q in sub_questions:
        content = retrieved_docs.get(sub_q, "")

        if not content:
            logger.info("No content — FAILED: '%s'", sub_q)
            failed.append(sub_q)
            continue

        try:
            result = _grade_chain.invoke({"question": sub_q, "context": content})
            if result.binary_score.strip().lower() == "yes":
                logger.info("RELEVANT: '%s'", sub_q)
                passed.append(sub_q)
            else:
                logger.info("NOT RELEVANT: '%s'", sub_q)
                failed.append(sub_q)
        except Exception as e:
            # Conservative: grading errors count as failures
            logger.error("Grading error for '%s': %s — marking failed.", sub_q, e)
            failed.append(sub_q)

    logger.info("Grading complete — passed: %d, failed: %d", len(passed), len(failed))

    # ── Step 3: write results to state ────────────────────────────────────────
    # route_after_grade will read these values to decide the next node.
    return {
        "failed_questions": failed,
        "retrieved_docs":   retrieved_docs,
    }


# ── Edge function: route_after_grade  (FIX 1 — pure routing, no state writes) ─
def route_after_grade(state: AgentState) -> Literal["generate", "rewrite", "__end__"]:
    """
    Pure routing function — reads state written by the grade node and returns
    the name of the next node.

    This function must NOT modify state. It only reads and returns a string.

    Routing table:
    ┌──────────────────────────────────────────┬──────────────────────────────┐
    │ Condition                                │ Route                        │
    ├──────────────────────────────────────────┼──────────────────────────────┤
    │ All sub-questions passed grading         │ generate                     │
    │ Some/all failed + rewrites remaining     │ rewrite (failed only)        │
    │ Some failed + ceiling hit                │ generate (partial context)   │
    │ All failed + ceiling hit                 │ __end__ (no hallucination)   │
    └──────────────────────────────────────────┴──────────────────────────────┘
    """
    failed:        list[str] = state.get("failed_questions", [])
    sub_questions: list[str] = state.get("sub_questions", [])
    rewrite_count: int       = state.get("rewrite_count", 0)

    # All passed → generate
    if not failed:
        logger.info("All sub-questions passed → generate")
        return "generate"

    # Failures exist but rewrite budget remains → rewrite
    if rewrite_count < MAX_REWRITES:
        logger.info(
            "Routing to rewrite (attempt %d/%d) for %d failed sub-question(s)",
            rewrite_count + 1, MAX_REWRITES, len(failed),
        )
        return "rewrite"

    # Rewrite ceiling hit
    passed_count = len(sub_questions) - len(failed)
    if passed_count > 0:
        logger.info(
            "Ceiling hit — partial context (%d/%d passed) → generate",
            passed_count, len(sub_questions),
        )
        return "generate"

    logger.info("Ceiling hit — zero relevant docs → END (avoiding hallucination)")
    return "__end__"



# ── Node 5: rewrite ───────────────────────────────────────────────────────────
def rewrite(state: AgentState) -> dict:
    """
    Rewrite only the sub-questions that failed grading.

    Sub-questions that already retrieved relevant docs are left untouched in
    retrieved_docs so their results carry forward to generation.

    State changes:
        sub_questions    ← failed sub-questions replaced with rewritten versions
        failed_questions ← cleared (will be repopulated by grade_documents)
        rewrite_count    ← incremented
    """
    logger.info("---REWRITE---")

    failed:         list[str] = state.get("failed_questions", [])
    sub_questions:  list[str] = state.get("sub_questions", [])
    rewrite_count:  int       = state.get("rewrite_count", 0)
    retrieved_docs: dict      = state.get("retrieved_docs", {})

    if not failed:
        logger.warning("Rewrite node called but failed_questions is empty.")
        return {
            "sub_questions":    sub_questions,
            "failed_questions": [],
            "rewrite_count":    rewrite_count + 1,
        }

    # Build a set for quick lookup
    failed_set = set(failed)

    # Rewrite each failed sub-question
    rewritten_map: dict[str, str] = {}
    for sub_q in failed:
        try:
            new_q = _rewrite_chain.invoke({"question": sub_q}).strip()
            if not new_q:
                raise ValueError("Rewrite chain returned empty string.")
            rewritten_map[sub_q] = new_q
            logger.info("Rewritten: '%s' → '%s'", sub_q, new_q)
        except Exception as e:
            logger.error("Rewrite failed for '%s': %s. Keeping original.", sub_q, e)
            rewritten_map[sub_q] = sub_q  # keep original if rewrite fails

    # Replace failed sub-questions in the master list; preserve order and successes
    updated_sub_questions: list[str] = []
    for q in sub_questions:
        if q in failed_set:
            updated_sub_questions.append(rewritten_map[q])
            # Migrate any partial content under the new key
            if q in retrieved_docs:
                retrieved_docs[rewritten_map[q]] = retrieved_docs.pop(q)
        else:
            updated_sub_questions.append(q)  # successful — keep unchanged

    return {
        "sub_questions":    updated_sub_questions,
        "failed_questions": [],         # cleared — grade_documents will repopulate
        "rewrite_count":    rewrite_count + 1,
        "retrieved_docs":   retrieved_docs,
    }


# ── Node 6: generate ──────────────────────────────────────────────────────────
def generate(state: AgentState) -> dict:
    """
    Produce a final answer from all retrieved context.

    Combines retrieved_docs across all sub-questions into a structured context
    block so the LLM can see which evidence maps to which sub-question.
    Sub-questions with no retrieved content are noted explicitly so the LLM
    can acknowledge gaps rather than hallucinate answers.

    State changes:
        messages ← final answer appended
    """
    logger.info("---GENERATE---")

    original_question: str       = state["messages"][0].content
    sub_questions:     list[str] = state.get("sub_questions", [])
    retrieved_docs:    dict      = state.get("retrieved_docs", {})

    if not sub_questions:
        logger.error("Generate called with no sub-questions in state.")
        return {"messages": ["I was unable to retrieve any information to answer your question."]}

    # Build a structured context block: one section per sub-question
    context_sections: list[str] = []
    for i, sub_q in enumerate(sub_questions, 1):
        content = retrieved_docs.get(sub_q, "")
        if content:
            context_sections.append(
                f"Sub-question {i}: {sub_q}\n"
                f"Retrieved context:\n{content}"
            )
        else:
            context_sections.append(
                f"Sub-question {i}: {sub_q}\n"
                f"Retrieved context: [No relevant documents found]"
            )

    context = "\n\n" + ("\n\n" + "─" * 60 + "\n\n").join(context_sections)

    try:
        response = _rag_chain.invoke({
            "context":  context,
            "question": original_question,
        })
    except Exception as e:
        logger.error("RAG chain failed during generate: %s", e)
        response = "I encountered an error while generating the answer. Please try again."

    logger.info("Generation complete.")
    return {"messages": [response]}



In [21]:

# =============================================================================
# 7. Build the LangGraph workflow
# =============================================================================

builder = StateGraph(AgentState)

# Register nodes
builder.add_node("decompose", decompose)
builder.add_node("agent",     agent)
builder.add_node("retriever", _tool_node)
builder.add_node("grade",     grade)
builder.add_node("rewrite",   rewrite)
builder.add_node("generate",  generate)

# ── Edges ──────────────────────────────────────────────────────────────────────

# Entry point always goes to decompose first
builder.add_edge(START, "decompose")

# After decompose, always go to agent
builder.add_edge("decompose", "agent")

# Agent decides whether to call a tool or end
builder.add_conditional_edges(
    "agent",
    tools_condition,
    {
        "tools": "retriever",   # tool_call present → retrieve
        END: END,           # no tool_call → agent decided to end directly
    },
)

# Retriever → grade (always — grade is now a node, not an edge function)
builder.add_edge("retriever", "grade")
# Grade → route (FIX 1 — route_after_grade is the pure edge function,
#                grade is the node that wrote the state it reads from)
builder.add_conditional_edges(
    "grade",
    route_after_grade,                     # pure routing, no state writes
    {"generate": "generate", "rewrite": "rewrite", "__end__": END},
)
# After rewrite, go back to agent for another retrieval pass
builder.add_edge("rewrite",  "agent")

# After generate, we're done
builder.add_edge("generate", END)

# ── Compile ────────────────────────────────────────────────────────────────────
try:
    graph = builder.compile()
    logger.info("Graph compiled successfully.")
except Exception as e:
    logger.exception("Failed to compile LangGraph.")
    raise


2026-02-19 21:26:51,329 [INFO] __main__ — Graph compiled successfully.
2026-02-19 21:26:51,329 [INFO] Graph compiled successfully.


In [22]:

# =============================================================================
# 8. Entry point
# =============================================================================

def run(question: str) -> str:
    """
    Public entry point for the agentic RAG pipeline.

    Args:
        question: The user's question (simple or complex).

    Returns:
        The final answer string.

    Raises:
        ValueError: If question is empty.
        RuntimeError: If the graph fails to produce a final answer.
    """
    if not question or not question.strip():
        raise ValueError("Question must not be empty.")

    logger.info("Running pipeline for question: '%s'", question)

    initial_state: AgentState = {
        "messages":          [HumanMessage(content=question)],
        "sub_questions":     [],
        "retrieved_docs":    {},
        "failed_questions":  [],
        "rewrite_count":     0,
    }

    try:
        result = graph.invoke(initial_state)
    except Exception as e:
        logger.exception("Graph execution failed.")
        raise RuntimeError(f"Pipeline failed: {e}") from e

    # Extract the final answer — last message in state
    messages = result.get("messages", [])
    if not messages:
        raise RuntimeError("Graph returned no messages.")

    final = messages[-1]
    answer = final.content if hasattr(final, "content") else str(final)

    logger.info("Pipeline complete.")
    return answer

In [23]:


if __name__ == "__main__":
    questions = [
        "What is LangGraph and how does it differ from LangChain?",        # multi-part
        "How do I build a chatbot with memory using LangChain?",           # single
        "What are LangGraph workflows and how do I use map-reduce in it?", # multi-part
    ]

    for q in questions:
        print(f"\n{'='*70}")
        print(f"Question: {q}")
        print("=" * 70)
        try:
            answer = run(q)
            print(f"Answer:\n{answer}")
        except Exception as e:
            print(f"[ERROR] {e}")


2026-02-19 21:26:51,346 [INFO] __main__ — Running pipeline for question: 'What is LangGraph and how does it differ from LangChain?'
2026-02-19 21:26:51,346 [INFO] Running pipeline for question: 'What is LangGraph and how does it differ from LangChain?'
2026-02-19 21:26:51,355 [INFO] __main__ — ---DECOMPOSE---
2026-02-19 21:26:51,355 [INFO] ---DECOMPOSE---



Question: What is LangGraph and how does it differ from LangChain?


2026-02-19 21:26:52,245 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-19 21:26:52,253 [INFO] __main__ — Decomposed into 3 sub-question(s): ['What is LangGraph?', 'What is LangChain?', 'How do LangGraph and LangChain differ from each other?']
2026-02-19 21:26:52,253 [INFO] Decomposed into 3 sub-question(s): ['What is LangGraph?', 'What is LangChain?', 'How do LangGraph and LangChain differ from each other?']
2026-02-19 21:26:52,254 [INFO] __main__ — ---CALL AGENT---
2026-02-19 21:26:52,254 [INFO] ---CALL AGENT---
2026-02-19 21:26:53,099 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-19 21:26:53,109 [INFO] __main__ — Agent processed sub-question: 'What is LangGraph?'
2026-02-19 21:26:53,109 [INFO] Agent processed sub-question: 'What is LangGraph?'
2026-02-19 21:26:53,814 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-19 21:26:53,822 [INFO] __mai

Answer:
[Chunk 1] Score: 0.38 | Source: https://python.langchain.com/docs/tutorials/
​ Core benefits
Standard model interfaceDifferent providers have unique APIs for interacting with models, including the format of responses. LangChain standardizes how you interact with models so that you can seamlessly swap providers and avoid lock-in.Learn moreEasy to use, highly flexible agentLangChain’s agent abstraction is designed to be easy to get started with, letting you build a simple agent in under 10 lines of code. But it also provides enough flexibility to allow you to do all the context engineering your heart desires.Learn moreBuilt on top of LangGraphLangChain’s agents are built on top of LangGraph. This allows us to take advantage of LangGraph’s durable execution, human-in-the-loop support, persistence, and more.Learn moreDebug with LangSmithGain deep visibility into complex agent behavior with visualization tools that trace execution paths, capture state transitions, and provide detail

2026-02-19 21:27:13,548 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-19 21:27:13,553 [INFO] __main__ — Decomposed into 5 sub-question(s): ['What is LangChain and how does it relate to building chatbots?', 'What are the components of a chatbot that enable it to have memory?', 'How can LangChain be utilized to implement memory in a chatbot?', 'What are the steps to set up a basic chatbot using LangChain?', 'How can I test and refine a chatbot with memory built using LangChain?']
2026-02-19 21:27:13,553 [INFO] Decomposed into 5 sub-question(s): ['What is LangChain and how does it relate to building chatbots?', 'What are the components of a chatbot that enable it to have memory?', 'How can LangChain be utilized to implement memory in a chatbot?', 'What are the steps to set up a basic chatbot using LangChain?', 'How can I test and refine a chatbot with memory built using LangChain?']
2026-02-19 21:27:13,556 [INFO] __main__ — ---CALL AGENT---

Answer:
Building a chatbot with memory using LangChain involves integrating conversational memory functionalities to handle multi-turn interactions and preserve context across threads. LangChain is an open-source framework that facilitates this by providing pre-built agent architectures and integration with large language models. To implement memory management in a chatbot, LangChain supports adding both short and long-term memory, enabling the chatbot to maintain context over extended interactions. 

The installation and configuration steps involve setting up LangChain and potentially integrating it with deployment services like LangSmith for efficient deployment. Unfortunately, the specific technical components or step-by-step instructions for installing and configuring a chatbot with memory using LangChain were not retrieved. To assess and enhance chatbot performance, consider deploying the application with LangSmith, which offers tools for improved responsiveness and structured out

2026-02-19 21:27:55,369 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-19 21:27:55,378 [INFO] __main__ — Decomposed into 2 sub-question(s): ['What are LangGraph workflows?', 'How is map-reduce used in LangGraph workflows?']
2026-02-19 21:27:55,378 [INFO] Decomposed into 2 sub-question(s): ['What are LangGraph workflows?', 'How is map-reduce used in LangGraph workflows?']
2026-02-19 21:27:55,379 [INFO] __main__ — ---CALL AGENT---
2026-02-19 21:27:55,379 [INFO] ---CALL AGENT---
2026-02-19 21:27:56,086 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-19 21:27:56,096 [INFO] __main__ — Agent processed sub-question: 'What are LangGraph workflows?'
2026-02-19 21:27:56,096 [INFO] Agent processed sub-question: 'What are LangGraph workflows?'
2026-02-19 21:27:56,800 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-19 21:27:56,820 [INFO] __main__ — Agent proc

Answer:
No relevant documents found in the local vector store.
